# runtime

> Execute code and edit values in the live kernel namespace

In [ ]:
#| default_exp runtime

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import paar.runtime as R
from paar.runtime import run, ExecResult, set_value

class _Shell:
    "Minimal IPython stand-in whose run_cell execs into a shared user_ns."
    def __init__(self): self.user_ns = {}
    def run_cell(self, code, store_history=True):
        import types
        ns = self.user_ns
        err = None; result = None
        try:
            block = compile(code, '<t>', 'exec')
            exec(block, ns)
            # emulate IPython binding `_` for a trailing expression
            import ast
            body = ast.parse(code).body
            if body and isinstance(body[-1], ast.Expr):
                result = eval(compile(ast.Expression(body[-1].value), '<t>', 'eval'), ns)
                ns['_'] = result
        except Exception as e:
            err = e
        return types.SimpleNamespace(result=result, error_in_exec=err,
                                     error_before_exec=None, success=err is None)

def test_run_global_assignment_mutates_ns():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('x = 5')
    assert isinstance(r, ExecResult) and r.ok is True
    assert r.result is None and sh.user_ns['x'] == 5   # assignment: no result row
def test_run_global_expression_makes_result_row():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('3 + 4')
    assert r.ok is True and r.result is not None
    assert r.result.name == 'result' and r.result.value == '7' and r.result.accessor == ('_',)
def test_run_global_modify_existing():
    sh = _Shell(); sh.user_ns['n'] = 1; R.get_ipython = lambda: sh
    run('n = n + 41'); assert sh.user_ns['n'] == 42
def test_run_error_is_captured():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('1/0')
    assert r.ok is False and 'ZeroDivisionError' in r.error
def test_run_no_ipython():
    R.get_ipython = lambda: None
    r = run('x=1'); assert r.ok is False and 'no IPython' in r.error
def test_run_isolated_no_leak():
    sh = _Shell(); sh.user_ns['seed'] = 10; R.get_ipython = lambda: sh
    r = run('y = seed + 1\ny', scope='isolated')
    assert r.ok is True and r.result is not None and r.result.value == '11'
    assert 'y' not in sh.user_ns                       # isolated: no leak into globals
    assert r.result.is_container is False and r.result.accessor == ()   # flat, non-expandable
def test_run_isolated_statement_only():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('z = 1', scope='isolated')
    assert r.ok is True and r.result is None and 'z' not in sh.user_ns

def test_set_value_toplevel():
    sh = _Shell(); sh.user_ns['x'] = 1; R.get_ipython = lambda: sh
    assert set_value(('x',), '99') is None and sh.user_ns['x'] == 99
def test_set_value_dict_item():
    sh = _Shell(); sh.user_ns['d'] = {'k': 1}; R.get_ipython = lambda: sh
    assert set_value(('d', 0), '2') is None and sh.user_ns['d']['k'] == 2   # accessor ('d',0)->d['k']
def test_set_value_list_element():
    sh = _Shell(); sh.user_ns['L'] = [10, 20]; R.get_ipython = lambda: sh
    assert set_value(('L', 1), '21') is None and sh.user_ns['L'][1] == 21
def test_set_value_bad_target_errors():
    sh = _Shell(); sh.user_ns['s'] = {1, 2}; R.get_ipython = lambda: sh
    err = set_value(('s', 0), '9')          # set element path is display-only, not an lvalue
    assert err is not None and 's' in sh.user_ns   # error string, no crash

for t in [test_run_global_assignment_mutates_ns, test_run_global_expression_makes_result_row,
          test_run_global_modify_existing, test_run_error_is_captured, test_run_no_ipython,
          test_run_isolated_no_leak, test_run_isolated_statement_only,
          test_set_value_toplevel, test_set_value_dict_item, test_set_value_list_element,
          test_set_value_bad_target_errors]: t()

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass
import ast
from paar.core import VarInfo
from paar.snapshot import _var_info, _walk
from paar.providers import value_str
from IPython.utils.capture import capture_output
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

@dataclass
class ExecResult:
    "Outcome of running code: last value as an inspector row, captured stdout, and error text."
    ok: bool
    result: VarInfo|None = None
    stdout: str = ''
    error: str|None = None

def _fmt_err(res):
    "Formatted exception text from a run_cell result, or None."
    e = getattr(res, 'error_in_exec', None) or getattr(res, 'error_before_exec', None)
    return f'{type(e).__name__}: {e}' if e is not None else None

def _exec_capture(code, ns):
    "Exec statements in ns; return the value of a trailing expression, else None."
    block = ast.parse(code)
    value = None
    if block.body and isinstance(block.body[-1], ast.Expr):
        last = ast.Expression(block.body.pop().value)
        exec(compile(block, '<paar>', 'exec'), ns)
        value = eval(compile(last, '<paar>', 'eval'), ns)
    else:
        exec(compile(block, '<paar>', 'exec'), ns)
    return value

def _flat_result(val):
    "A non-expandable result row (isolated mode leaves nothing reachable in user_ns)."
    return VarInfo(name='result', type=type(val).__name__, value=value_str(val))

def run(code, scope='global'):
    "Run `code` in the kernel; scope='global' mutates user_ns, 'isolated' uses a scratch copy."
    ip = get_ipython()
    if ip is None: return ExecResult(ok=False, error='no IPython kernel')
    if scope == 'isolated':
        ns = dict(ip.user_ns)
        try:
            with capture_output() as cap:
                val = _exec_capture(code, ns)
            return ExecResult(ok=True, result=None if val is None else _flat_result(val), stdout=cap.stdout)
        except Exception as e:
            return ExecResult(ok=False, error=f'{type(e).__name__}: {e}')
    with capture_output() as cap:
        res = ip.run_cell(code, store_history=True)
    err = _fmt_err(res)
    result = _var_info('result', res.result, ('_',), '_') if res.result is not None else None
    return ExecResult(ok=bool(res.success) and err is None, result=result, stdout=cap.stdout, error=err)

def set_value(accessor, expr):
    "Assign `expr` (evaluated in user_ns) to the lvalue at `accessor`; return error text or None."
    ip = get_ipython()
    if ip is None: return 'no IPython kernel'
    try: _, path = _walk(ip.user_ns, tuple(accessor))
    except Exception as e: return f'bad accessor: {e}'
    try:
        exec(f'{path} = ({expr})', ip.user_ns)
        return None
    except Exception as e:
        return f'{type(e).__name__}: {e}'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()